# Data Exploration - Hybrid AI Complaint Escalation System
This notebook explores the synthetic datasets used to train each model component.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('Imports successful')

## 1. Classical ML Dataset

In [ ]:
# Generate the same dataset used in training
from train_classical_model import generate_synthetic_data

df = generate_synthetic_data(n_samples=1200)
print('Shape:', df.shape)
print('\nColumn names:', list(df.columns))
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution:')
print(df['risk_label'].value_counts())

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class distribution
df['risk_label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Count')

# Previous complaints by risk
df.boxplot(column='previous_complaints', by='risk_label', ax=axes[1])
axes[1].set_title('Previous Complaints by Risk Level')
axes[1].set_xlabel('Risk Level')

# Order value by risk
df.boxplot(column='order_value', by='risk_label', ax=axes[2])
axes[2].set_title('Order Value by Risk Level')
axes[2].set_xlabel('Risk Level')

plt.suptitle('')
plt.tight_layout()
plt.savefig('exploration_classical.png', dpi=100)
plt.show()

## 2. RNN Dataset

In [ ]:
from train_rnn_model import generate_text_data, tokenise

texts, labels = generate_text_data(n_samples=1500)
print('Number of samples:', len(texts))
print('Label distribution:', Counter(labels))

text_lengths = [len(t.split()) for t in texts]
print(f'\nText length stats:')
print(f'  Min: {min(text_lengths)}  Max: {max(text_lengths)}  Mean: {np.mean(text_lengths):.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_counts = Counter(labels)
axes[0].bar(['Low (0)', 'Medium (1)', 'High (2)'],
            [label_counts[0], label_counts[1], label_counts[2]],
            color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('RNN Dataset Label Distribution')

# Text length distribution
axes[1].hist(text_lengths, bins=20, color='#3498db', edgecolor='white')
axes[1].axvline(50, color='red', linestyle='--', label='Max len (50 tokens)')
axes[1].set_title('Complaint Text Length Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('exploration_rnn.png', dpi=100)
plt.show()

## 3. CNN Dataset (Fashion-MNIST)

In [ ]:
import tensorflow as tf

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Pixel range:', X_train.min(), '-', X_train.max())
print('Classes:', CLASS_NAMES)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[i], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Sample Images (one per class)')
plt.tight_layout()
plt.savefig('exploration_cnn.png', dpi=100)
plt.show()

## 4. Train / Validation / Test Split Summary

In [ ]:
split_summary = pd.DataFrame({
    'Component': ['Classical ML', 'RNN/LSTM', 'CNN'],
    'Total Samples': [1200, 1500, 70000],
    'Train (70%)': [840, 1050, 49000],
    'Validation (15%)': [180, 225, 9000],
    'Test (15%)': [180, 225, 10000],
})
split_summary

## 5. Correlation Analysis (Classical ML Features)

In [ ]:
label_num = df['risk_label'].map({'low': 0, 'medium': 1, 'high': 2})
acc_num   = (df['account_type'] == 'premium').astype(int)
corr_df   = pd.DataFrame({
    'risk_label': label_num,
    'previous_complaints': df['previous_complaints'],
    'order_value': df['order_value'],
    'tenure': df['customer_tenure_years'],
    'is_premium': acc_num,
})

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('exploration_correlation.png', dpi=100)
plt.show()

print('\nKey finding: previous_complaints has the strongest correlation with risk_label.')